# Lesson 2.8 — Scratch notebook: environment, trajectory, and the expert planner

This is a **working notebook**, not a curated lesson. It keeps the path by which the
project found out how to produce a successful PickCube demonstration, including the
wrong turns. Everything here was written in three passes:

1. **Environment fundamentals** (2.8.1–2.8.2) — build `PickCube-v1`, read the 42-d
   observation, sample and step an action.
2. **Random trajectory collection** — assemble a transition list and a full episode.
3. **Hunting for a planner** — search the installed ManiSkill package for the
   demonstration generator, pivot to `pd_joint_pos`, and construct the planner.

> **Note on ordering.** The three import self-checks at the end of the original file
> (`mplib`, `sapien`, the planner class) have been **moved up** to sit before the planner
> construction they support. Their code is unchanged and their recorded outputs are their
> originals, so `execution_count` values now appear out of order. Cell sources and outputs
> were not otherwise modified.

## Why `pd_joint_pos` appears halfway through

The first pass uses `pd_joint_delta_pos`, which is the control mode of the project's
dataset fixture. The planner later rejects that: its gripper helpers emit
`[qpos(7), gripper]`, an 8-d **absolute** position action, so a delta-mode environment
fails with `Received action of shape torch.Size([15]) but expected shape (1, 8)`.
The switch to `pd_joint_pos` in the second half is that discovery being recorded.

> **Environment requirement.** Constructing the planner requires `mplib`, which is built
> against the NumPy 1.x C API. In an environment with NumPy 2.x this notebook's planner
> cell kills the kernel with `SIGSEGV` at address `0x0`. Run the planner cells under
> `embodied310` (NumPy 1.26.4); see `notes/progress.md`.

A curated version of the successful path lives in
`2.9_expert_demonstrations.ipynb`; the batch collector is
`scripts/generate_expert_demo.py`.

## 2.8.1 — Build the environment

Three arguments decide what the agent sees and does:

- `obs_mode="state"` — the flattened robot + object state vector, no images.
- `control_mode="pd_joint_delta_pos"` — the action is a **joint position change**, which
  is the mode the project's dataset fixture uses.
- `render_mode="rgb_array"` — works without a display server in a notebook.

In [2]:
import gymnasium as gym

import mani_skill.envs


# --------------------------------------------------
# Create PickCube environment
#
# obs_mode:
#   state = robot + object information
#
# control_mode:
#   pd_joint_delta_pos
#   means action controls joint position change
#
# render_mode:
#   rgb_array works better in notebook
# --------------------------------------------------

env = gym.make(
    "PickCube-v1",
    obs_mode="state",
    control_mode="pd_joint_delta_pos",
    render_mode="rgb_array",
)


print("Environment initialized")

print("Observation:")
print(env.observation_space)

print("Action:")
print(env.action_space)

Environment initialized
Observation:
Box(-inf, inf, (1, 42), float32)
Action:
Box(-1.0, 1.0, (8,), float32)


In [3]:
# --------------------------------------------------
# Reset simulator
#
# This creates:
# - physics scene
# - robot
# - objects
# - task state
# --------------------------------------------------

obs, info = env.reset()


print("Reset successful!")

print("Observation shape:")
print(obs.shape)

Reset successful!
Observation shape:
torch.Size([1, 42])


### Reset and read the 42-d observation

`env.reset()` builds the physics scene, the robot, the objects, and the task state. The
returned tensor is `(num_envs, 42)`: the leading axis exists because ManiSkill is
vectorised, and `num_envs=1` here.

`obs_mode="state"` returns one flat vector, so the structure behind those 42 numbers has
to be peeled back by hand. The verified layout is

```text
[0:9]   qpos
[9:18]  qvel
[18:19] is_grasped
[19:26] tcp_pose
[26:29] goal_pos
[29:36] obj_pose
[36:39] tcp_to_obj_pos
[39:42] obj_to_goal_pos
```

In [4]:
# Print the raw observation tensor
#
# obs shape:
# (num_envs, state_dimension)
#
# Here:
# num_envs = 1
# state_dimension = 42


print(obs)

print("-------------------")

print("dtype:")
print(obs.dtype)

print("-------------------")

print("device:")
print(obs.device)

tensor([[ 0.0076,  0.3909, -0.0461, -1.9406, -0.0307,  2.3389,  0.8057,  0.0400,
          0.0400,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0058, -0.0273,  0.1793,  0.0199,  0.9997,
         -0.0170,  0.0036,  0.0530,  0.0357,  0.2286, -0.0760, -0.0574,  0.0200,
          0.1607, -0.0000, -0.0000, -0.9870, -0.0818, -0.0301, -0.1593,  0.1290,
          0.0931,  0.2086]])
-------------------
dtype:
torch.float32
-------------------
device:
cpu


In [5]:
# Convert torch tensor to numpy array
#
# detach:
# remove gradient tracking
#
# cpu:
# move data from GPU to CPU
#
# numpy:
# convert to numpy format


state = (
    obs
    .detach()
    .cpu()
    .numpy()
)


print(state.shape)

(1, 42)


In [6]:
def analyze_pickcube_state(state):
    """
    Analyze PickCube-v1 state observation.

    Current observation contains:
    - robot joint information
    - end-effector pose
    - object pose
    - goal information
    - gripper state

    This helps us understand:
    what information a robot policy receives.
    """

    # Remove environment dimension
    # shape:
    # (1,42) -> (42,)

    state = state[0]


    print("Total dimensions:")
    print(len(state))


    print("\n========== Robot Joint Position ==========")

    # First 9 dimensions:
    # robot joint position information

    print(state[:9])


    print("\n========== Robot Joint Velocity ==========")

    # Next 9 dimensions:
    # robot joint velocity

    print(state[9:18])


    print("\n========== End Effector Pose ==========")

    # Position + orientation

    print(state[18:25])


    print("\n========== Object Pose ==========")

    # Cube position and orientation

    print(state[25:32])


    print("\n========== Goal ==========")

    # Target position

    print(state[32:35])


    print("\n========== Relative Information ==========")

    # Relative distance information

    print(state[35:41])


    print("\n========== Gripper ==========")

    # Open / close state

    print(state[41])

In [7]:
analyze_pickcube_state(state)

Total dimensions:
42

========== Robot Joint Position ==========
[ 0.00755627  0.39089483 -0.04611887 -1.9406402  -0.03071309  2.3389194
  0.8057291   0.04        0.04      ]

========== Robot Joint Velocity ==========
[0. 0. 0. 0. 0. 0. 0. 0. 0.]

========== End Effector Pose ==========
[ 0.          0.0057635  -0.02726336  0.17932712  0.01989457  0.9996503
 -0.01704317]

========== Object Pose ==========
[ 0.00363891  0.05301899  0.03570371  0.22857928 -0.07601617 -0.05735836
  0.02      ]

========== Goal ==========
[ 0.1606978 -0.        -0.       ]

========== Relative Information ==========
[-0.9870037  -0.08177967 -0.030095   -0.15932712  0.12903516  0.09306207]

========== Gripper ==========
0.20857929


## 2.8.2 — Actions and the step loop

`env.action_space.sample()` draws a uniformly random action from `[-1, 1]^8`. This is a
**pipeline fixture**, not a policy: the resulting trajectory has valid structure and
essentially no useful behaviour.

`env.step(action)` returns five values. Two of them are separate stop conditions and are
easy to conflate:

- `terminated` — the task ended in success or unrecoverable failure;
- `truncated` — the episode hit a step or time limit, which says nothing about success.

In [8]:
# Sample one random action

action = env.action_space.sample()

print("Action:")
print(action)

print("----------------")

print("Shape:")
print(action.shape)

Action:
[ 0.31634188 -0.9451699   0.25665474 -0.64098585 -0.9949915  -0.92318964
  0.04182364 -0.71126246]
----------------
Shape:
(8,)


### Note on the duplicated sampling cell

The cell above and the cell below both call `env.action_space.sample()`. The first one
is the minimal version; the second repeats it with the shape inspection spelled out.
The duplication is kept as the original record — the content is identical.

In [9]:
# --------------------------------------------------
# Sample one action from ManiSkill action space
#
# Action represents:
# "what the robot should do at this timestep"
#
# Current control mode:
# pd_joint_delta_pos
#
# Therefore:
# action = desired joint position change
# --------------------------------------------------


action = env.action_space.sample()


print("Action:")
print(action)


print("------------------------")


print("Action shape:")
print(action.shape)

Action:
[-0.97418123  0.1994947   0.4745303  -0.02078718  0.26149392 -0.4823804
 -0.17995225 -0.9917719 ]
------------------------
Action shape:
(8,)


Both cells sample one action from the same space, so the shape printed here is the
action contract: `(8,)` in `[-1, 1]`, reshaped to `(8,)` from the vectorised
`Box(-1.0, 1.0, (8,), float32)`.

In [10]:
# --------------------------------------------------
# Execute one robot action
#
# env.step(action):
#
# input:
#     action_t
#
# output:
#     next observation
#     reward
#     termination signal
# --------------------------------------------------


next_obs, reward, terminated, truncated, info = env.step(action)


print("Next observation:")
print(next_obs.shape)


print("----------------")


print("Reward:")
print(reward)


print("----------------")


print("Info:")
print(info)

Next observation:
torch.Size([1, 42])
----------------
Reward:
tensor([0.0591])
----------------
Info:
{'elapsed_steps': tensor([1], dtype=torch.int32), 'success': tensor([False]), 'is_obj_placed': tensor([False]), 'is_robot_static': tensor([False]), 'is_grasped': tensor([False])}


## 2.8.3 — Collect a random episode

The two blocks below build the same thing at different levels of care: first a bare
20-step list of `(observation, action, reward)`, then a full episode that also records
the success flag and stops on `terminated` / `truncated`.

That list-of-dicts **is** the minimal robot dataset format. Everything downstream
(HDF5, LeRobot, DataLoader) is bookkeeping around it.

The stored observation is the state **before** the action was applied, which is what
fixes the `(o_t, a_t)` pairing convention.

### Why `terminated` and `truncated` are both checked

Stopping only on `terminated` would run past the time limit; stopping only on
`truncated` would ignore an early success. The loop breaks on either, which is what
makes the recorded length meaningful.

Note the pairing convention this loop establishes: the observation appended at step
`t` is the state **entered** at that step, so the stored list is
`(o_t, a_t)` — not `(o_t, a_{t+1})`.

### Minimal transition list

The smallest useful container: three parallel lists. It records exactly what the loop
consumes and produces, and nothing else — no success flag, no stop condition.

In [11]:
# --------------------------------------------------
# Collect one short trajectory
#
# We store:
# observation
# action
# reward
#
# This is the minimal robot dataset format
# --------------------------------------------------


trajectory = {
    "observations": [],
    "actions": [],
    "rewards": []
}


# Reset environment

obs, info = env.reset()


for step in range(20):

    # Random action for now
    # Later replaced by expert action

    action = env.action_space.sample()


    next_obs, reward, terminated, truncated, info = env.step(action)


    # Store current transition

    trajectory["observations"].append(
        obs.cpu().numpy()
    )


    trajectory["actions"].append(
        action
    )


    trajectory["rewards"].append(
        reward
    )


    obs = next_obs


    if terminated or truncated:
        break



print(
    "Collected steps:",
    len(trajectory["actions"])
)

Collected steps: 20


### Reading the result honestly

What comes out here is a **random** episode: the total reward is small, and no episode
in this notebook succeeds. That is expected and it is the point — this trajectory
validates the pipeline, it cannot train a policy.

The signal that separates random from expert data is action smoothness, not reward:
random actions jump by `mean |Δa| ≈ 0.67` between steps, expert planner actions by
`≈ 0.0078`. See `2.9_expert_demonstrations.ipynb`.

In [12]:
print(trajectory["rewards"])
print("Total reward:",
      sum(trajectory["rewards"]))

[tensor([0.0497]), tensor([0.0477]), tensor([0.0461]), tensor([0.0463]), tensor([0.0442]), tensor([0.0354]), tensor([0.0342]), tensor([0.0379]), tensor([0.0408]), tensor([0.0395]), tensor([0.0356]), tensor([0.0325]), tensor([0.0349]), tensor([0.0389]), tensor([0.0411]), tensor([0.0501]), tensor([0.0480]), tensor([0.0379]), tensor([0.0353]), tensor([0.0341])]
Total reward: tensor([0.8100])


### Full episode with a stop condition

Same idea, but it also stores `success` and stops on `terminated` / `truncated`, so the
returned length reflects a real episode boundary rather than a fixed step budget.

In [13]:
# --------------------------------------------------
# Collect one complete episode
#
# We record:
# - observation
# - action
# - reward
# - success
#
# This format is closer to real robot datasets
# --------------------------------------------------


def collect_random_episode(env, max_steps=200):

    trajectory = {
        "observations": [],
        "actions": [],
        "rewards": [],
        "success": False,
    }


    obs, info = env.reset()


    for step in range(max_steps):

        # Random policy
        action = env.action_space.sample()


        next_obs, reward, terminated, truncated, info = env.step(action)


        # Save transition

        trajectory["observations"].append(
            obs.cpu().numpy()
        )

        trajectory["actions"].append(
            action
        )

        trajectory["rewards"].append(
            reward.cpu().numpy()
        )


        obs = next_obs


        if terminated or truncated:

            trajectory["success"] = (
                info["success"].item()
            )

            break


    return trajectory

In [14]:
random_traj = collect_random_episode(env)

print(
    "Length:",
    len(random_traj["actions"])
)

print(
    "Success:",
    random_traj["success"]
)

Length: 50
Success: False


## 2.8.4 — Hunting for the demonstration generator

ManiSkill 3 changed its APIs, so the planner is located by inspecting the **installed
package** rather than trusting an older tutorial. The searches below walk the
`mani_skill` tree for demonstration files, agents, motion-planning modules, and finally
the `PickCube` implementation itself.

Reading `motionplanner.py` directly is the point: it is the expert controller that
generates demonstration trajectories, so its interface defines what a successful
episode looks like.

In [15]:
import os
import mani_skill


# --------------------------------------------------
# Get ManiSkill installation directory
#
# We need this path to inspect:
# - demos
# - examples
# - utilities
# --------------------------------------------------

mani_skill_path = os.path.dirname(
    mani_skill.__file__
)


print("ManiSkill path:")
print(mani_skill_path)

ManiSkill path:
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill


The cell above computed `mani_skill_path` from the imported module, so this search
follows the installed package wherever it actually lives. That is the portable way to
locate ManiSkill internals — unlike the hard-coded paths two cells below.

In [16]:
# --------------------------------------------------
# Search files related to demonstrations
#
# ManiSkill version 3 changed some APIs,
# so we inspect the installed package
# instead of assuming old tutorials.
# --------------------------------------------------

for root, dirs, files in os.walk(mani_skill_path):

    for file in files:

        if "demo" in file.lower():

            print(
                os.path.join(root, file)
            )

/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_vis_textures.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_vis_pcd.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_robot.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_random_action.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_reset_distribution.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_manual_control_continuous.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_vis_segmentation.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/demo_manual_control.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/__pycach

In [17]:
# --------------------------------------------------
# Search robot agents
#
# ManiSkill uses agents to represent:
# - robot model
# - controller
# - action interface
# --------------------------------------------------

for root, dirs, files in os.walk(mani_skill_path):

    for file in files:

        if "agent" in file.lower():

            print(
                os.path.join(root, file)
            )

/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/base_agent.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/base_real_agent.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/multi_agent.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/__pycache__/multi_agent.cpython-312.pyc
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/__pycache__/base_agent.cpython-312.pyc
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/agents/__pycache__/base_real_agent.cpython-312.pyc


In [18]:
# --------------------------------------------------
# Search motion planning related modules
#
# Expert demonstrations in simulation are usually
# generated by planners.
# --------------------------------------------------

keywords = [
    "motion",
    "planner",
    "planning"
]


for root, dirs, files in os.walk(mani_skill_path):

    for file in files:

        filename = file.lower()

        for keyword in keywords:

            if keyword in filename:

                print(
                    os.path.join(root, file)
                )

                break

/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/xarm6/motionplanner.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/xarm6/__pycache__/motionplanner.cpython-312.pyc
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/two_finger_gripper/motionplanner.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/two_finger_gripper/__pycache__/motionplanner.cpython-312.pyc
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/panda/motionplanner.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/panda/motionplanner_stick.py
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/panda/__pycache__/motionplanner_stick.cpython

In [19]:
# --------------------------------------------------
# Find PickCube implementation
# --------------------------------------------------

for root, dirs, files in os.walk(mani_skill_path):

    for file in files:

        if "pickcube" in file.lower():

            print(
                os.path.join(root, file)
            )

### Trap: these paths are absolute and environment-specific

Both inspections below open files by a **hard-coded absolute path** that embeds the
`embodied` environment's Python version:

```text
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/...
```

That works on this machine and breaks on any other Python version, environment name, or
install location. It is also why the second path in the following cell carries a second
hard-coded prefix. Prefer deriving the path from the module, as the search cells above
already do:

```python
path = Path(mani_skill.__file__).parent / "examples" / "motionplanning" / "..."
```

In [20]:
# --------------------------------------------------
# Inspect Panda motion planner
#
# This is the expert controller used for generating
# demonstration trajectories.
# --------------------------------------------------

planner_path = (
    "/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/"
    "mani_skill/examples/motionplanning/panda/motionplanner.py"
)


with open(planner_path, "r") as f:
    planner_code = f.read()


print(planner_code[:4000])

import mplib
import numpy as np
import sapien

from mani_skill.envs.sapien_env import BaseEnv
from mani_skill.examples.motionplanning.two_finger_gripper.motionplanner import TwoFingerGripperMotionPlanningSolver


class PandaArmMotionPlanningSolver(TwoFingerGripperMotionPlanningSolver):
    OPEN = 1
    CLOSED = -1
    MOVE_GROUP = "panda_hand_tcp"

    def __init__(
        self,
        env: BaseEnv,
        debug: bool = False,
        vis: bool = True,
        base_pose: sapien.Pose = None,  # TODO mplib doesn't support robot base being anywhere but 0
        visualize_target_grasp_pose: bool = True,
        print_env_info: bool = True,
        joint_vel_limits=0.9,
        joint_acc_limits=0.9,
    ):
        super().__init__(env, debug, vis, base_pose, visualize_target_grasp_pose, print_env_info, joint_vel_limits, joint_acc_limits)


In [21]:
planner_path = "/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/mani_skill/examples/motionplanning/two_finger_gripper/motionplanner.py"

with open(planner_path, "r") as f:
    code = f.read()

print(code[:8000])

import mplib
import numpy as np
import sapien

from mani_skill.envs.sapien_env import BaseEnv
from mani_skill.envs.scene import ManiSkillScene
from mani_skill.examples.motionplanning.base_motionplanner.motionplanner import BaseMotionPlanningSolver
from transforms3d import quaternions


class TwoFingerGripperMotionPlanningSolver(BaseMotionPlanningSolver):
    OPEN = 1
    CLOSED = -1

    def __init__(
        self,
        env: BaseEnv,
        debug: bool = False,
        vis: bool = True,
        base_pose: sapien.Pose = None,  # TODO mplib doesn't support robot base being anywhere but 0
        visualize_target_grasp_pose: bool = True,
        print_env_info: bool = True,
        joint_vel_limits=0.9,
        joint_acc_limits=0.9,
    ):
        super().__init__(env, debug, vis, base_pose, print_env_info, joint_vel_limits, joint_acc_limits)
        self.gripper_state = self.OPEN
        self.visualize_target_grasp_pose = visualize_target_grasp_pose
        self.grasp_pose_visual = N

## 2.8.5 — Motion planning requires `pd_joint_pos`

This is where the first pass runs into the constraint.

`env.unwrapped` strips the Gymnasium wrapper to reach the ManiSkill API — the agent, the
robot, its links and joints, and the base pose the planner needs. On the delta-mode
environment, the planner's action contract cannot be satisfied.

The `pd_joint_pos` environment is created twice below (once to inspect, once for the
planning run). That duplication is kept: it is the original working record, and the
second block is the one the planner is bound to.

In [2]:
# ==========================================
# Step 0:
# Create ManiSkill environment
# ==========================================

import gymnasium as gym
import mani_skill.envs


env = gym.make(
    "PickCube-v1",
    obs_mode="state",
    control_mode="pd_joint_delta_pos",
    render_mode="rgb_array",
)


print("Environment created")
print(type(env))

Environment created
<class 'mani_skill.utils.registration.TimeLimitWrapper'>


In [3]:
# ==========================================
# Step 1:
# Remove Gymnasium wrapper
# ==========================================


real_env = env.unwrapped


print("Environment:")
print(type(real_env))


print("----------------")


print("Robot agent:")
print(type(real_env.agent))

Environment:
<class 'mani_skill.envs.tasks.tabletop.pick_cube.PickCubeEnv'>
----------------
Robot agent:
<class 'mani_skill.agents.robots.panda.panda.Panda'>


In [5]:
# ==========================================
# Step 2:
# Inspect Panda robot
# ==========================================


robot = real_env.agent.robot


print("Robot:")
print(robot)


print("----------------")


print("Robot attributes:")
print(
    [x for x in dir(robot) if "pose" in x.lower()]
)

Robot:
<panda: struct of type <class 'mani_skill.utils.structs.articulation.Articulation'>; managing 1 <class 'sapien.pysapien.physx.PhysxArticulation'> objects>
----------------
Robot attributes:
['get_pose', 'get_root_pose', 'initial_pose', 'pose', 'root_pose', 'set_pose', 'set_root_pose']


In [6]:
# ==========================================
# Step 3:
# Get robot base pose
# ==========================================


robot_base_pose = robot.pose


print(robot_base_pose)

Pose(raw_pose=tensor([[-6.1500e-01,  7.2760e-11, -1.4901e-08,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00]]))


In [7]:
print(real_env.control_mode)

pd_joint_delta_pos


The comment below — *"Motion planner outputs joint positions"* — is the whole reason
this second environment exists. It is the moment the `pd_joint_delta_pos` assumption
from 2.8.1 breaks.

In [8]:
import gymnasium as gym
import mani_skill.envs


env = gym.make(
    "PickCube-v1",
    obs_mode="state",

    # Important:
    # Motion planner outputs joint positions.
    # This cell only inspects the mode; the planner is bound to the env created below.
    control_mode="pd_joint_pos",

    render_mode="rgb_array",
)


print(env)

<TimeLimitWrapper<OrderEnforcing<PickCubeEnv<PickCube-v1>>>>


In [9]:
real_env = env.unwrapped

print(real_env.control_mode)

pd_joint_pos


### Role of each of the two `pd_joint_pos` environments

- the cell just above **inspects** the mode (`print(real_env.control_mode)`);
- the cell below is the one the planner is actually **bound to**, and it is the one
  whose `real_env` is used for `reset`, `robot_base_pose`, and the planner constructor.

They set identical options, and the duplication is preserved as the original record.
When re-running the notebook, keep track of which `env` the later cells close over.

In [11]:
# ==================================================
# Create PickCube environment for Motion Planning
#
# Important:
# Motion Planner outputs joint positions.
#
# Therefore:
# control_mode = pd_joint_pos
#
# ==================================================

import gymnasium as gym
import mani_skill.envs


env = gym.make(
    "PickCube-v1",

    # We first use state observation
    # because motion planner does not need images
    obs_mode="state",

    # Important:
    # Planner outputs joint position targets.
    # This is the environment the planner is bound to.
    control_mode="pd_joint_pos",

    render_mode="rgb_array",
)


print("Environment created")

print(type(env))

print("----------------")

print("Control mode:")
print(env.unwrapped.control_mode)

Environment created
<class 'mani_skill.utils.registration.TimeLimitWrapper'>
----------------
Control mode:
pd_joint_pos


In [12]:
# ==================================================
# Remove Gym wrapper
# ==================================================

real_env = env.unwrapped


print(type(real_env))

print(real_env.agent)

<class 'mani_skill.envs.tasks.tabletop.pick_cube.PickCubeEnv'>


In [13]:
# ==================================================
# Reset environment
#
# The planner needs:
# - current robot state
# - cube position
# - goal position
# ==================================================

obs, info = real_env.reset()


print("Observation:")
print(obs.shape)

print("----------------")

print(info)

Observation:
torch.Size([1, 42])
----------------
{'elapsed_steps': tensor([0], dtype=torch.int32), 'success': tensor([False]), 'is_obj_placed': tensor([False]), 'is_robot_static': tensor([True]), 'is_grasped': tensor([False]), 'reconfigure': False}


In [14]:
robot_base_pose = real_env.agent.robot.pose

print(robot_base_pose)

Pose(raw_pose=tensor([[-6.1500e-01,  7.2760e-11, -1.4901e-08,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00]]))


## 2.8.6 — Build the expert planner

Before constructing anything, confirm the pieces import. These three cells were at the
end of the original file; they are moved here because they are a prerequisite,
not a conclusion.

`PandaArmMotionPlanningSolver` is the expert controller. It is constructed against the
**unwrapped** environment and is given the robot base pose, because `mplib` assumes the
base is at the origin unless told otherwise.

> **This is the cell that requires NumPy 1.x.** Running it under NumPy 2.x terminates the
> kernel with a segmentation fault inside `mplib`, which cannot be caught by
> `try` / `except`. Under `embodied310` (NumPy 1.26.4) it builds successfully.

In [1]:
import mplib
print("mplib ok")

mplib ok


In [2]:
import sapien
print("sapien ok")

sapien ok


In [3]:
from mani_skill.examples.motionplanning.panda.motionplanner import PandaArmMotionPlanningSolver
print("planner class ok")

planner class ok


In [16]:
# ==========================================
# Import Panda Motion Planner
#
# This class provides the expert planner
# for Panda robot.
# ==========================================

from mani_skill.examples.motionplanning.panda.motionplanner import (
    PandaArmMotionPlanningSolver
)


print("PandaArmMotionPlanningSolver imported!")

PandaArmMotionPlanningSolver imported!


In [ ]:
import time


print("Start creating planner...")

start = time.time()


planner = PandaArmMotionPlanningSolver(
    real_env,

    # No extra debug information
    debug=False,

    # Disable visualization first
    vis=False,

    # Robot base coordinate
    base_pose=robot_base_pose
)


end = time.time()


print("----------------")
print("Planner created!")
print(
    "Initialization time:",
    end-start,
    "seconds"
)